# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema, accessible at the following URL:
<br>
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata object
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n{metadata.description}")

## 2. Data Overview
Explore available record sets, fields, and their `@id`s. All entity references will use their `@id` as per the Croissant schema.

In [ ]:
# List all record sets defined in the dataset
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    print('Available record sets:')
    for rs in metadata.record_sets:
        print(f"- @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")
else:
    # Sometimes 'recordSet' or similar is used
    if hasattr(metadata, 'recordSet') and metadata.recordSet:
        print('Available record sets:')
        for rs in metadata.recordSet:
            print(f"- @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")
    else:
        print('No record sets found in the metadata.')

### Explore Example Record: Field and Column `@id`s
We'll enumerate some records, fields, and column IDs for a selected record set.

In [ ]:
# Try to infer at least one record set to use
# For this dataset, if there are no explicit record sets, fallback to 'records()' default

record_sets_to_try = []

# Try to extract all available record set IDs
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    # mlcroissant 0.3+ style
    record_sets_to_try = [rs['@id'] for rs in metadata.record_sets]
elif hasattr(metadata, 'recordSet') and metadata.recordSet:
    # Some Croissant variants use 'recordSet'
    record_sets_to_try = [rs['@id'] for rs in metadata.recordSet]

# If none, just show record examples using the default loader
if record_sets_to_try:
    # Render field/column-level info for the first record set
    record_set_id = record_sets_to_try[0]
    print(f'Example records for record set @id: {record_set_id}')
    for i, rec in enumerate(dataset.records(record_set=record_set_id)):
        print(f'Record {i+1}:', rec)
        if i > 2:
            break
else:
    print('No record sets available; printing a few example records from the default records():')
    for i, rec in enumerate(dataset.records()):
        print(f'Record {i+1}:', rec)
        if i > 2:
            break

## 3. Data Extraction
Load data from the available record set(s) into Pandas DataFrames for processing. 
We will use only the record set(s) and field `@id`s discovered above.

In [ ]:
# Extract all records from each available record set

# Recompute record_sets variable for use in the next cells
record_sets = []
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_sets = [rs['@id'] for rs in metadata.record_sets]
elif hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = [rs['@id'] for rs in metadata.recordSet]

dataframes = {}
if record_sets:
    for record_set_id in record_sets:
        print(f'Loading record set: {record_set_id}')
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
else:
    # Fallback: load all records using records() if record sets are not specified in the schema
    print('No explicit record sets, loading top-level records...')
    all_records = list(dataset.records())
    df = pd.DataFrame(all_records)
    record_sets = ['default']
    dataframes['default'] = df

# Show available DataFrames and columns for each
for rs_id in dataframes.keys():
    print(f"\nRecordSet @id: {rs_id}")
    print("Columns:", dataframes[rs_id].columns.tolist())

# Display first 5 records for the first record set
example_record_set = record_sets[0]
dataframes[example_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the data: filter numeric fields, normalize, and group. All field references use their `@id` as column names as loaded from Croissant.

In [ ]:
# Choose a record set and numeric field for demonstration (replace with IDs you observe in your dataset)

# Use the first record set and one of its numeric fields, e.g., 'MW' (Molecular Weight), which is common in proteomics
record_set_id = example_record_set
df = dataframes[record_set_id]

# List available columns that may be numeric
print(f"Available fields in record set {record_set_id}:")
for col in df.columns:
    print(f"- {col}: {df[col].dtype}")

# Try picking a numeric column, e.g. 'MW' or 'molecular_weight' or similar
numeric_field_id = None
for candidate in ['MW', 'molecular_weight', 'Molecular Weight', 'molecularWeight', 'cr:MW', 'cr:molecular_weight', 'mass', 'cr:mass']:
    if candidate in df.columns:
        numeric_field_id = candidate
        break
if numeric_field_id is None:
    # Fallback: pick the first float/int column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}")
    # Drop records with missing values to allow numeric operations
    filtered_df = df.dropna(subset=[numeric_field_id])
    try:
        # Convert to float if necessary
        filtered_df[numeric_field_id] = filtered_df[numeric_field_id].astype(float)
    except Exception:
        pass
    threshold = filtered_df[numeric_field_id].mean()
    filtered_df2 = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):\n", filtered_df2.head())

    # Normalize field
    filtered_df2[f"{numeric_field_id}_normalized"] = (
        filtered_df2[numeric_field_id] - filtered_df2[numeric_field_id].mean()
    ) / filtered_df2[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id}:\n", filtered_df2[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a likely categorical field
    possible_group_fields = ['description', 'cr:description', 'accession', 'cr:accession', 'Sample', 'sample', 'cr:sample']
    group_field = None
    for gf in possible_group_fields:
        if gf in df.columns:
            group_field = gf
            break

    if group_field:
        print(f"\nGrouping by {group_field} and taking mean of numeric fields...")
        grouped_df = (
            filtered_df2.groupby(group_field)[numeric_field_id].mean().reset_index()
        )
        print(grouped_df.head())
else:
    print('No numeric field found for analysis.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(10,4))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field_id].dropna().astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.subplot(1, 2, 2)
    if f"{numeric_field_id}_normalized" in filtered_df2.columns:
        sns.histplot(filtered_df2[f"{numeric_field_id}_normalized"], bins=30, kde=True)
        plt.title(f"Normalized {numeric_field_id}")
        plt.xlabel(f"{numeric_field_id}_normalized")
    plt.tight_layout()
    plt.show()
else:
    print('No numeric field identified for visualization.')

## 6. Conclusion
In this notebook, we've used `mlcroissant` to load and explore the FAIR^2 mass spectrometry dataset described by its Croissant schema (@id references throughout). We:
- Inspected available record sets and fields by `@id`.
- Loaded records into Pandas DataFrames for flexible analysis.
- Performed basic filtering and normalization on a selected numeric field.
- Visualized the data distributions.

**Next steps:** The dataset contains advanced proteomics fields for use in downstream machine learning tasks, biomarker discovery, and biological analysis. You can extend this notebook by exploring more columns, performing deeper analyses, or exporting derived datasets.